# Configure

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Packages
import requests
import os
import re
from os.path import join
from pathlib import Path
import yaml
from yaml.loader import SafeLoader
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
import rioxarray as rio

import unsafe.download as undown
import unsafe.files as unfile
import unsafe.unzip as ununzip
import unsafe.exp as unexp
import unsafe.ddfs as unddf
import unsafe.ensemble as unens

In [3]:
# Name the fips, statefips, stateabbr, and nation that
# we are using for this analysis
# We pass these in as a list even though the framework currently
# processes a single county so that it can facilitate that
# expansion in the future
# TODO - could make sense to define these in the future
# in json or other formats instead of as input in code
fips_args = {
    'FIPS': ['42101'], 
    'STATEFIPS': ['42'],
    'STATEABBR': ['PA'],
    'NATION': ['US']
}
FIPS = fips_args['FIPS'][0]
NATION = fips_args['NATION'][0]

In [4]:
# We need to pass in a config file that sets up
# constants and the structure for downlading data
# For the directory structure of our case study, 
# we use the following 
ABS_DIR = Path().absolute().parents[0]

CONFIG_FILEP = join(ABS_DIR, 'config', 'config.yaml')
# Open the config file and load
with open(CONFIG_FILEP) as f:
    CONFIG = yaml.load(f, Loader=SafeLoader)

# Wildcards for urls
URL_WILDCARDS = CONFIG['url_wildcards']

# Get the file extensions for api endpoints
API_EXT = CONFIG['api_ext']

# Get the CRS constants
NSI_CRS = CONFIG['nsi_crs']

# Dictionary of ref_names
REF_NAMES_DICT = CONFIG['ref_names']

# Dictionary of ref_id_names
REF_ID_NAMES_DICT = CONFIG['ref_id_names']

# Coefficient of variation
# for structure values
COEF_VARIATION = CONFIG['coef_var']

# First floor elevation dictionary
FFE_DICT = CONFIG['ffe_dict']

# Number of states of the world
N_SOW = CONFIG['sows']

# Data for flood depth grids
# Get hazard model variables
HAZ_FILEN = CONFIG['haz_filename']
# Get CRS for depth grids
HAZ_CRS = CONFIG['haz_crs']
# Ensemble members
HAZ_NENS = CONFIG['haz_nens']
# Number of columns for each depth grid
HAZ_NCOLS = CONFIG['haz_ncols']
# Num rows for each depth grid
HAZ_NROWS = CONFIG['haz_nrows']
# Lower left x coordinate
HAZ_XLL = CONFIG['haz_xll']
# Lower left y coordinate
HAZ_YLL = CONFIG['haz_yll']
# Cell resolution
HAZ_RES = CONFIG['haz_res']
# NODATA values
HAZ_NODATA = CONFIG['haz_nodata']

# Get the files we need downloaded
DOWNLOAD = pd.json_normalize(CONFIG['download'], sep='_').T

# We can also specify the filepath to the
# raw data directory
FR = join(ABS_DIR, "data", "raw")

# And external - where our hazard data should be
FE = join(FR, "external")

# Set up interim and results directories as well
# We already use "FR" for raw, we use "FO" 
# because you can also think of results
# as output
FI = join(ABS_DIR, "data", "interim")
FO = join(ABS_DIR, "data", "results")

# "Raw" data directories for exposure, vulnerability (vuln) and
# administrative reference files
EXP_DIR_R = join(FR, "exp")
VULN_DIR_R = join(FR, "vuln")
REF_DIR_R = join(FR, "ref")
# Haz is for depth grids
HAZ_DIR_R = join(FE, "haz")
# Pol is for NFHL
POL_DIR_R = join(FR, "pol")

# Unzip directory 
UNZIP_DIR = join(FR, "unzipped")

# We want to process unzipped data and move it
# to the interim directory where we keep
# processed data
# Get the filepaths for unzipped data
# We unzipped the depth grids (haz) and 
# ddfs (vuln) into the "external"/ subdirectory
HAZ_DIR_UZ = join(UNZIP_DIR, "external", "haz")
POL_DIR_UZ = join(UNZIP_DIR, "pol")
REF_DIR_UZ = join(UNZIP_DIR, "ref")
VULN_DIR_UZ = join(UNZIP_DIR, "external", "vuln")

# "Interim" data directories
EXP_DIR_I = join(FI, "exp")
VULN_DIR_I = join(FI, "vuln")
REF_DIR_I = join(FI, "ref")
# Haz is for depth grids
HAZ_DIR_I = join(FI, "haz")
# Pol is for NFHL
POL_DIR_I = join(FI, "pol")

# Download and unzip data

In [5]:
wcard_dict = {x: fips_args[x[1:-1]][0] for x in URL_WILDCARDS}
undown.download_raw(DOWNLOAD, wcard_dict,
                    FR, API_EXT)

Downloaded from: https://nsi.sec.usace.army.mil/nsiapi/structures?fips=42101
Downloaded from: https://phl.carto.com/api/v2/sql?filename=opa_properties_public&format=geojson&skipfields=cartodb_id&q=SELECT+*+FROM+opa_properties_public
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/ab9e89e1273f445bb265846c90b38a96_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/84baed491de44f539889f2af178ad85c_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://hazards.fema.gov/nfhlv2/output/County/420757_20230701.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_42_tract.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_42_bg.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TABBLOCK20/tl_2022_42_tabblock20.zip
Downloaded from: https://static-data-screeningtool.geoplatform.gov/data-versions/1.0/data/score/download

In [6]:
ununzip.unzip_raw(FR, UNZIP_DIR)

Unzipped: nfhl
Unzipped: zcta
Unzipped: county
Unzipped: bg
Unzipped: tract
Unzipped: block
Unzipped: ddfs
Unzipped: RIFT_domain
Unzipped: Irene


# Prepare data for ensemble

The study domain corresponds to 12 digit USGS hydrological unit code (HUC) watershed 020402031008. We will spatially merge the NSI structures and Philadelphia data to this extent. We will restrict the other downloaded geospatial data to objects that intersect with this (e.g., Census Tracts that overlap). We may clip these for plotting purposes later.

## Study area boundary

In [5]:
CLIP_SHP_FILEP = join(HAZ_DIR_UZ, 'RIFT_domain', 'domain_1.shp')
clip_geo = gpd.read_file(CLIP_SHP_FILEP)

/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


## Process exposure

We will start by subsetting several datasets to the envelope of our clip polygon and then do the processing on those subsets.

### Get subsets of NSI and Philly data

In [6]:
# Load in the NSI and Philly assessor, parcel, and footprint data
nsi_gdf = unexp.get_nsi_geo(FIPS, NSI_CRS, EXP_DIR_R)

assess_cols = ['assessment_date', 'basements', 'building_code',
               'building_code_description', 'building_code_description_new',
               'category_code', 'category_code_description', 'census tract',
               'exterior_condition', 'garage_type', 'general_construction',
               'interior_condition','location', 'market_value',
               'market_value_date', 'number_stories', 'owner_1',
               'parcel_number', 'sale_date', 'sale_price',
               'quality_grade', 'taxable_building', 'exempt_building',
               'total_area', 'total_livable_area',
               'topography', 'unit', 'year_built',
               'other_building', 'garage_type',
               'year_built_estimate', 'zoning']
assess = gpd.read_file(join(EXP_DIR_R, FIPS, 'assess.geojson'),
                       mask=clip_geo, columns=assess_cols)

parcel = gpd.read_file(join(EXP_DIR_R, FIPS, 'parcel.geojson'),
                       mask=clip_geo)
bld_fp = gpd.read_file(join(EXP_DIR_R, FIPS, 'bldfp.geojson'),
                       mask=clip_geo)

Prepared geodataframe


/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)
/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


#### NSI subset

We'll follow existing UNSAFE functions to get our NSI dataset. We're going to include any residential structure in occupancy type RES1 and RES3

In [7]:
# Set the values that we pass into the get_struct_subset function
occtype_list=['RES1-1SNB', 'RES1-2SNB', 'RES1-1SWB', 'RES1-2SWB',
              'RES1-SLNB', 'RES1-SLWB', 'RES1-3SNB', 'RES1-3SWB',
              'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F']
sub_string = 'occtype.isin(@occtype_list)'
nsi_filt = unexp.get_struct_subset(nsi_gdf,
                                   filter=sub_string,
                                   occtype_list=occtype_list)

EXP_OUT_FILEP = join(EXP_DIR_I, FIPS, 'nsi_res.pqt')
unfile.prepare_saving(EXP_OUT_FILEP)

# Clip to our boundary to reduce file size
nsi_clip_out = gpd.clip(nsi_filt, clip_geo.to_crs(nsi_filt.crs))

# Write file
nsi_clip_out.to_parquet(EXP_OUT_FILEP, index=False)

# Helpful summaries
print('Total NSI structures: {}'.format(len(nsi_gdf)))
print('Total NSI res structures: {}'.format(len(nsi_filt)))
print('Total NSI res structures in study area: {}'.format(len(nsi_clip_out)))

Total NSI structures: 527752
Total NSI res structures: 476174
Total NSI res structures in study area: 96630


#### Philly data subsets

We want to use the assessment data to identify residential structures. Then we will subset the building footprints and parcels correspondingly. To match up records, we will link `assess['parcel_number']` to `parcel['BRT_ID']` to `bld_fp['PARCEL_ID_NUM']`.

Condos will require extra processing. From the Maps@Phila.gov email: “The buildings are matched via their centroid to the PWD Parcels for their parcelid, they could use the parcelid to connect to PWD Parcels, then use the BRT_ID field in the PWD Parcels to get to the OPA Tax Accounts.  This won’t be the cleanest solution for condos, but there’s no real system for handling those anywhere.  You can tell [redacted] she’s welcome to point out any mismatches she finds directly to me, I’ve worked with her before on other projects.” We describe the condo processing approach above the corresponding cell block.

We start by processing the assessor data. We use the `building_code_description` column to identify RES1 and RES3 mappings by sampling records and checking the properties in street view apps (Google and Philadelphia's own) and Philadelphia Properties/Atlas apps. Some building code descriptions appear to uniformly map to RES1 or RES3, but some are mixed. For example, some buildings are coded as twin row homes, which we consider RES3, but their neighbor was demolished so effectively the property is RES1. For our 'main' sample, we use building footprint processing (to identify detached footprints) and sq. ft. statistics on individual row homes to identify likely RES1. For sensitivity checks, we use majority mappings for building code descriptions to occupancy type (and a few other checks). 

Below we split the `building_code_description` column in a way that gives us reduced form information for a subset of structure types we can look through manually. 

In [8]:
def split_bld_code(bld_desc):

    """
    Split a building code description into tokens based on the first numeric value.

    This function takes a building code description and splits it into tokens where
    all text before the first numeric value becomes one token, and all subsequent
    words (including numeric values) become individual tokens.

    Parameters
    ----------
    bld_desc : str
        A string containing the building code description.
        Example: 'APT 2-4 UNITS 3.5 STY MAS'

    Returns
    -------
    list
        A list where the first element is all text before the first number (as one string),
        followed by all remaining words as individual elements.
        Example: ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
        If no numeric values are found, returns the entire description as a single element list.

    Examples
    --------
    >>> split_bld_code('APT 2-4 UNITS 3.5 STY MAS')
    ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
    
    >>> split_bld_code('DET W/GAR 2 STY MASONRY')
    ['DET W/GAR', '2', 'STY', 'MASONRY']
    """

    if bld_desc is None:
        return bld_desc

    # Split the building code description into words
    full_code = bld_desc.split()

    # Find the index of the first string with a number as first character
    first_num_idx = next((i for i, word in enumerate(full_code) if word[0].isdigit()), None)
    
    if first_num_idx is not None:
        # Join everything before the first number as one token
        prefix = ' '.join(full_code[:first_num_idx])
        # Keep remaining words as separate tokens
        remaining = full_code[first_num_idx:]
        return [prefix] + remaining
    else:
        return [' '.join(full_code)]

In [9]:
print('Total tax records in study area: {}'.format(len(assess)))

# We want to retain structures with a building code description
assess_sub = assess[assess['building_code_description'].notnull()].copy()
# split up the building code description field
assess_sub.loc[:, 'bld_code_split'] = assess_sub['building_code_description'].apply(split_bld_code)

# get the occupancy type code and the remaining token 
# into separate columns
assess_sub.loc[:, 'bld_type'] = assess_sub['bld_code_split'].apply(lambda x: x[0])
assess_sub.loc[:, 'bld_code_rest'] = assess_sub['bld_code_split'].apply(lambda x: x[1:])
# helpful to have the rest as a single string for some inspections
# can drop the last token though (usually foundation type)
assess_sub.loc[:, 'bld_code_rest_str'] = assess_sub['bld_code_rest'].apply(lambda x: ' '.join(x[:-1]))

# We want to subset to the category codes that may have res buildings
cat_codes = ['1', '2', '3', '14']
assess_sub = assess_sub[assess_sub['category_code'].str.strip().isin(cat_codes)]

# We do not want "VACANT" 
assess_sub = assess_sub[~assess_sub['bld_type'].str.contains('VACANT')]

# We can also drop anything with empty bld_code_rest_str
assess_non_res = assess_sub[assess_sub['bld_code_rest_str'] == '']
assess_sub = assess_sub[assess_sub['bld_code_rest_str'] != '']

print('Sample of tax records in study area: {}'.format(len(assess_sub)))

Total tax records in study area: 123586
Sample of tax records in study area: 106482


Below, we take the reduced form building codes to sample 10 properties (or the number of properties in the new code) for manual checking. We generated two of these files to allow for two analysts to check each others mappings and converge on processing rules for main and sensitivity analyses. We comment out the sample writing lines to avoid overwriting data generated in our analysis. The files we generated and coded are available for others to inspect. They may also generate new samples (change the file suffix). 

In [10]:
# sample a few records from each bld_type group
samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)
# write out the parcel numbers and a few other columns and start 
# checking the ddf pairing
check_cols = ['parcel_number', 'bld_type', 'bld_code_rest_str',
              'building_code', 'category_code_description', 'zoning']
# check_dir = join(EXP_DIR_I, 'check_records')
# file_suf = '020425.csv'
# check_filep = join(check_dir, 'check_codes_' + file_suf)
# unfile.prepare_saving(check_filep)
# samples[check_cols].to_csv(check_filep, index=False)

/var/folders/d2/g0h08s551zb2hz_ws2g4ggh400hbd0/T/ipykernel_57034/2605892050.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)


We don't want to use the parcel centroid as a way to link with the flood hazard. We want to use the building footprint. We have to link the assessor records to parcels and building footprints. We need the parcels dataset because that's how we can merge the building footprints in. 

First, we will drop the `bld_type` that we identified as not having any residential structures. The remaining records are our residential subset. 

In [11]:
# Identified manually by evaluating partial (but sometimes full) samples of unique bld_type
drop_bld_codes = ['HOTEL', 'PRIV GAR']
assess_res = assess_sub[~assess_sub['bld_type'].isin(drop_bld_codes)].copy()

print('Sample of res tax records in study area: {}'.format(len(assess_res)))

Sample of res tax records in study area: 106300


We also want to add the taxable and exempt building value for our structure value

In [12]:
assess_res['val_struct'] = assess_res['taxable_building'] + assess_res['exempt_building']

We should also subset based on acceptable exterior and interior condition
The [documentation](https://metadata.phila.gov/#home/datasetdetails/5543865f20583086178c4ee5/representationdetails/55d624fdad35c7e854cb21a4/?view_287_per_page=100&view_287_page=1) tells us for exterior condition:

7. VACANT – No occupancy. FHA, VA, FNMA signs may be on the property. Property has been secured with fresh plywood over doors and windows.
8. SEALED – Doors and windows have been covered over by plywood, tin, concrete block or stucco. No interior access.
9. STRUCTURALLY COMPROMISED, OPEN TO THE WEATHER - Some or no windows, no door or door open, evidence of past abuse by vandals such as graffiti, missing railings, deteriorated wood and metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding, bays, etc. Broken windows with blackened and charred interior.

For interior: 

6. Vacant – No occupancy. FHA, VA, FNMA signs may be on the property.
Property has been secured with fresh plywood over doors and windows.
7. Sealed / Structurally Compromised, Open to the Weather –
Doors and windows have been covered over by plywood, tin, concrete block or
stucco. No interior access. Some or no windows, no door or door open, evidence
of past abuse by vandals such as graffiti, missing railings, deteriorated wood and
metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding,
bays, etc. Broken windows with blackened and charred interior.


In [13]:
assess_res = assess_res[(~assess_res['interior_condition'].str.strip().isin(['6', '7'])) &
                        (~assess_res['exterior_condition'].str.strip().isin(['7', '8', '9']))].copy()
print('Sample of occupied res tax records in study area: {}'.format(len(assess_res)))

Sample of occupied res tax records in study area: 103995


Next, we need to aggregate condos (any record that has a not null unit which ends up being mostly condos) by address because it is only way to link the assessor records to building footprints. Each condo has a different parcel number but only one of these will link to the parcels dataset (and subsequently the building footprints data). So, we need to find the condo parcel numbers in the assessor data that are in the parcels data, then find the units with the same address as that in the assessor data and aggregate them on structure value and other fields (bld_code_rest_str will have number of stories data). Then, we will merge these aggregated records with the non condo records in the assessor data. This is the dataset we will merge with the building footprints data. We will check whether the records that don't successsfully link to building footprints correspond to things like vacant land. 

In [14]:
# We want a subset of the records we need to aggregate
# These are records with a unit and and for which
# only one of the records with a shared address links
# to the parcels data
assess_res['has_parcel_match'] = assess_res['parcel_number'].isin(parcel['BRT_ID'])
res_agg = assess_res[(assess_res['unit'].notnull()) |
                     (assess_res['bld_type'] == 'RES CONDO')].copy()
res_no_agg = assess_res[~assess_res['parcel_number'].isin(res_agg['parcel_number'])].copy()


With these separated, we have the following workflow to get the building footprints linked up to the assessor records. 

1. Processing res_no_agg
  * Those with parcel match can be linked directly to bld_fp. Make the proper attribute based merges
  * Those w/o cannot be directly linked. Spatially join these with parcel.
    - Subset 1: when linked to a parcel without BRT_ID, just use the PARCEL_ID directly for a link
    - Subset 2: when linked to a parcel with BRT_ID, aggregate the parcels (they share a building)
2. Processing res_agg
  * The subset with parcel match should be aggregated with records that have the same location then linked to their PARCEL_ID through the location/ADDRESS merge
  * The subset w/o should have a spatial join with parcel and we should figure out the properties to aggregate (probably those with same location but maybe other things to look for)

In [14]:
par_cols = ['BRT_ID', 'PARCEL_ID', 'ADDRESS', 'geometry']
bld_cols = ['BIN', 'PARCEL_ID_NUM', 'ADDRESS', 'geometry']
tax_cols = ['parcel_number', 'bld_type', 'bld_code_rest',
            'number_stories', 'basements']

# 1. Processing residential properties without units
# Direct matches through assess[parcel number] to parcel[BRT_ID]
# Then parcel[PARCEL_ID] to bld_fp[PARCEL_ID_NUM]
direct_matches = res_no_agg[res_no_agg['has_parcel_match']].merge(
    parcel[par_cols].drop(columns=['geometry']),
    left_on='parcel_number',
    right_on='BRT_ID'
).drop(columns='geometry').copy()

direct_matches_final = bld_fp[bld_cols].merge(
   direct_matches,
    right_on='PARCEL_ID',
    left_on='PARCEL_ID_NUM'
)

# Spatial matching for properties without direct parcel match
spatial_candidates = res_no_agg[~res_no_agg['has_parcel_match']]
spatial_matches = gpd.sjoin(spatial_candidates,
    parcel[par_cols],
    how='left').drop(columns='geometry')

# Properties matched to parcels without BRT_ID
no_brt_matches = spatial_matches[spatial_matches['BRT_ID'].isnull()]
no_brt_matches = no_brt_matches[no_brt_matches['PARCEL_ID'].notnull()]
no_brt_matches_final = bld_fp[bld_cols].merge(
    no_brt_matches,
    right_on='PARCEL_ID',
    left_on='PARCEL_ID_NUM'
)

# Properties matched to parcels with BRT_ID (requiring aggregation)
brt_matches = spatial_matches[spatial_matches['BRT_ID'].notnull()]
# Aggregate records sharing a building
agg_dict = {col: 'first' for col in tax_cols}
agg_dict['val_struct'] = 'sum'
brt_matches_agg = brt_matches.groupby('PARCEL_ID').agg(agg_dict).reset_index()
brt_matches_final = bld_fp[bld_cols].merge(
    brt_matches_agg,
    right_on='PARCEL_ID',
    left_on='PARCEL_ID_NUM'
)

We need to do a bunch of post processing on these datasets in order. For starters, there are a number of tax records linked to multiple building footprints now. That's expected for apartments and we will disaggregate attributes into structures. Something else to deal with is properties with garages and sheds. Finally, we have to deal with records who are not linked to a building footprint becuase the building footprint and parcel boundary don't overlap in a way that allows for the assessor's point in polygon method to capture the intersections correctly. We'll work our way backwards through these needs. 

We're going to get the tax records that are linked to a parcel but not to building footprints. There are a number of records linked to two footprints and it's clear that we're not talking about a garage. What we need to do is for each property w/ a parcel link we couldn't link to a building footprint, find its parcel neighbor and see which one has two footprints. Assign the footprint that intersects with the one missing a footprint and remove its association with the neighboring parcel. 

In [313]:
direct_no_bld = direct_matches[~direct_matches['BRT_ID'].isin(direct_matches_final['BRT_ID'])]
par_no_bld = parcel[parcel['BRT_ID'].isin(direct_no_bld['BRT_ID'])]

# Start by getting a full spatial join of parcels without bld
# and bld footprints
par_bld_guess = gpd.sjoin(par_no_bld, bld_fp, how='left', predicate='intersects')

# Remove any entries with PARCEL_ID_NUM that is in direct_matches_final
# *and* that entry in direct_matches_final is only linked to one bld_fp
matches_one_bld = direct_matches_final[~direct_matches_final.duplicated('PARCEL_ID_NUM')]
par_bld_possible = par_bld_guess[~par_bld_guess['PARCEL_ID_NUM'].isin(matches_one_bld['PARCEL_ID_NUM'])]

# par_neighbs = gpd.sjoin(par_no_bld, parcel, how='inner', predicate='touches')

#### Testing bld_fp linking without centroid matches

Most of the assessment records get matched to building footprints successfully but there are a few inconsistencies because of the parcel boundaries not overlapping with the building enough for the centroid method to work. There are a number of records matched to multiple footprints (the entire footprint) in a way that makes it ambiguous to know the structure footprint for the record. This is an experimental section to see if we can get the residential structure for each record better than we can using the existing linkages. This will include some assumption-driven processing about how to filter for garages and other appurtenant structures that will have analogues in more of a post-processing step for the existing linked data. Will probably treat this as an experimental notebook that demonstrates the results of the comparisons since for clarity in the main analysis we'll want to have the assumptions baked in for more readability.   

In [388]:
# Start by overlaying the bld_fp with parcels
# Most records have direct links to assess_res through parcel_number/BRT_ID
# These are the ones we want to overlay - we will do links for
# nonmatched afterwards (may require condo aggregation)

par_cols = ['BRT_ID', 'PARCEL_ID', 'ADDRESS', 'geometry']
bld_cols = ['BIN', 'PARCEL_ID_NUM', 'ADDRESS', 'MAX_HGT', 'geometry']
tax_cols = ['parcel_number', 'bld_type', 'bld_code_rest',
            'building_code_description_new',
            'val_struct',
            'basements', 'unit']

assess_linked = assess_res[assess_res['has_parcel_match']].copy()
assess_no_link = assess_res[~assess_res['has_parcel_match']].copy()

direct_matches = parcel[par_cols].merge(
    assess_linked[tax_cols],
    right_on='parcel_number',
    left_on='BRT_ID'
)

# tax records are uniquely linked to parcels unless
# they refer to condos, in which case we only want
# to bring those tax records in later for aggregating
# things like structure value and then dividing across
# building footprints on the parcel
# in cases where this is only one building footprint, 
# you'd just keep what you aggregated
pc_res_dir = gpd.GeoDataFrame(direct_matches,
                              geometry=direct_matches['geometry'],
                              crs=parcel.crs)

bld_fp_o = gpd.overlay(pc_res_dir, bld_fp[bld_cols], how='intersection')

# Convert the new footprints to epsg 5070 for area calculations
bld_fp_o['m2_bld'] = bld_fp_o.to_crs(epsg='5070').area

# Drop links where area threshold not met
bld_fp_min_m2 = 10
bld_fp_drop = bld_fp_o.loc[bld_fp_o['m2_bld'] <= bld_fp_min_m2]
bld_fp_o = bld_fp_o.loc[bld_fp_o['m2_bld'] > bld_fp_min_m2]

# Bring back links where area threshold was not met
# but it's the only building reference available for the
# parcel
merge_back = pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])]['parcel_number']
merge_back_bld = bld_fp_drop[bld_fp_drop['parcel_number'].isin(merge_back)]
bld_fp_o = pd.concat([bld_fp_o, merge_back_bld], axis=0)

# print out the number of unmatched parcels
unmatched = len(pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Unmatched res tax records in study area: {}'.format(unmatched))
matched = len(pc_res_dir[pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Matched res tax records in study area: {}'.format(matched))
match_prop = (matched)/len(pc_res_dir)
print('Proportion of matched res tax records in study area: {}'.format(match_prop))

Unmatched res tax records in study area: 257
Matched res tax records in study area: 94016
Proportion of matched res tax records in study area: 0.9972738748103911


In [389]:
# Get a new id
# Records with identical geometry should have same building footprint id
# Because of direct_matches above, we will only have 1 record per
# group but this is a more generalizable solution than other options
bld_fp_o['geometry'] = bld_fp_o['geometry'].normalize()
bld_fp_o['bfid'] = bld_fp_o.groupby('geometry').ngroup()

# Calculate the number of parcels linked to this building footprint
bld_fp_o['n_parcels'] = bld_fp_o.groupby('bfid')['parcel_number'].transform('nunique')
# and vice versa
bld_fp_o['n_bld'] = bld_fp_o.groupby('parcel_number')['bfid'].transform('nunique')

We proceed with the following workflow:

1. Any building linked to one parcel gets assigned to that parcel.
2. When there are more than one buildings assigned to the parcel, combine those that touch and create a new bfid index. Recalculate the buildings in the parcel. And repeat step 1. 
3. For remaining cases, check the height and area of structures and compare to other records with the same building_code_description. Drop those that fall outside of the height threshold. Repeat step 1. 
4. For remaining cases, will require some combination of disaggregation of tax record to buildings and aggregating info from tax records before disaggregating to buildings. Note that some condos have 1:1 (get linked up in step 1 or 2 above) and still need to be linked to unmatched tax records for aggregation. 

In [390]:
# pc_bld will be our final dataframe of links
# we'll append processed subset dfs into a list
# and then concat into pc_bld when done
pc_bld_l = []

# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
par_one_bld_match = bld_fp_o[bld_fp_o['n_bld'] == 1]
pc_bld_l.append(par_one_bld_match)
par_one_bld_many = bld_fp_o[bld_fp_o['n_bld'] > 1]

# For more complex cases, combine touching footprints in 
# the same parcel and subset them out
check = par_one_bld_many.dissolve(by='parcel_number', 
                                  aggfunc={'m2_bld': 'max',
                                           'MAX_HGT': 'max',
                                           'bld_type': 'first',
                                           'bld_code_rest': 'first',
                                           'bfid': lambda x: list(x)},
                                  method='coverage').reset_index()
check_exp = check.explode()
check_exp['n_bld'] = check_exp.groupby('parcel_number').transform('size')
pc_bld_from_dissolve = check_exp[check_exp['n_bld'] == 1]
pc_bld_l.append(pc_bld_from_dissolve)

Remaining cases are either properties with garages/sheds/etc. or sould be treated more like apartment complexes (occasionally requiring aggregation of nonmatched condo units first). We can concat the two dataframes for parcel/bld matches and examine the building type and number of stories combos for their height and area distributions. We can use this to identify thresholds for dropping appurtenant structures and trying again for main building identification. We want to leverage when there are multiple remaining candidate footprints as well to see whether one is larger and taller than the others, which often signals main building for the types of structures we're looking at here. 

When these steps don't work at getting unique pairings, we'll flag the remaining records and treat them like apartment complexes (disaggregating the parcel record to numerous structures). As a sensitivity check, we can also remove apartment complexes, condos, and the records we treat like apartment complexes (they make the minority of our overall sample) and remove any NSI records that overlap with their parcel boundaries. 

In [391]:
check_cols = ['parcel_number', 'bld_type', 'bld_code_rest', 'MAX_HGT', 'm2_bld']
main_bld_ref = pd.concat([par_one_bld_match[check_cols],
                          pc_bld_from_dissolve[check_cols]], axis=0)
main_bld_ref['stories_n'] = main_bld_ref['bld_code_rest'].apply(lambda x: x[0])

# We are generally looking at min because we expect main buildings to be
# larger and taller
main_stats = main_bld_ref.groupby(['bld_type', 'stories_n'])[['MAX_HGT', 'm2_bld']].describe()
main_thresh = main_stats.iloc[:, main_stats.columns.get_level_values(1).isin(['min', '75%', 'std'])].copy()
# Want upper & lower bounds on height and lower bounds on area
main_thresh = main_thresh.iloc[:, [0, 1, 2, 4]]
main_thresh.columns = ['height_std', 'height_min', 'height_75', 'area_min']
main_thresh['height_upper'] = main_thresh['height_75'] + 2*main_thresh['height_std']
main_thresh = main_thresh[['height_min', 'height_upper', 'area_min']].copy()

# We will reuse the lower bound on area_min
# There are a few cases like parcel_number 888000038 that are new construction
# and the building footprint data isn't up to date with that. They get a really
# small sliver of the neighboring property but shouldn't
# We already linked these slivers, which is ok because they do represent
# new construction and now they are geolocated to a building location
# But we don't want these to skew our identification of main buildings
main_thresh.loc[main_thresh['area_min'] < bld_fp_min_m2, 'area_min'] = bld_fp_min_m2

In [392]:
# Remaining cases
pc_remain = bld_fp_o[~bld_fp_o['parcel_number'].isin(main_bld_ref['parcel_number'])]

# We can start by taking the subset where ADDRESS_1 is equal to ADDRESS_2 because
# these are cases where the building footprint centroid matched up
# to the parcel. While it won't reconcile garage issues, it will reconcile
# the issue where a large enough slice of a different building is hanging around
pc_remain_same_adr = pc_remain[pc_remain['ADDRESS_1'] == pc_remain['ADDRESS_2']].copy()
pc_remain_same_adr['n_bld'] = pc_remain_same_adr.groupby('parcel_number').transform('size')
pc_same_adr_keep = pc_remain_same_adr[pc_remain_same_adr['n_bld'] == 1]
# Add to our processed df list
pc_bld_l.append(pc_same_adr_keep)

# Drop the parcels from pc_remain_same_adr that are in pc_same_adr_keep
# and filter these with the thresholds from our main_bld_ref
pc_remain_rest = pc_remain[~pc_remain['parcel_number'].isin(pc_same_adr_keep['parcel_number'])].copy()
# Add stories_n to merge in the filters from above
pc_remain_rest['stories_n'] = pc_remain_rest['bld_code_rest'].apply(lambda x: x[0])
# Get the threshold values in
pc_remain_rest = pc_remain_rest.set_index(['bld_type', 'stories_n']).join(main_thresh)

# Get the subset of properties that meet height and area thresholds
pc_remain_main = pc_remain_rest[(pc_remain_rest['MAX_HGT'] >= pc_remain_rest['height_min']) &
                                (pc_remain_rest['MAX_HGT'] <= pc_remain_rest['height_upper']) &
                                (pc_remain_rest['m2_bld'] >= pc_remain_rest['area_min'])].copy()
pc_remain_main['n_bld'] = pc_remain_main.groupby('parcel_number').transform('size')
pc_meet_thresh_unique = pc_remain_main[pc_remain_main['n_bld'] == 1]
pc_bld_l.append(pc_meet_thresh_unique)

# Lastly, the parcels that meet "buest guess" thresholds
# let's see if we can identify the main structures or have to treat all
# as if they are apartment complexes
pc_meet_thresh = pc_remain_main[pc_remain_main['n_bld'] > 1].copy()

# compare the ratio of each building's m2_bld to the group
# of buildings in that parcel
pc_meet_thresh['area_ratio'] = (pc_meet_thresh['m2_bld'] / 
                                pc_meet_thresh.groupby('parcel_number')['m2_bld'].transform('max'))
pc_meet_thresh['height_ratio'] = (pc_meet_thresh['MAX_HGT'] / 
                                  pc_meet_thresh.groupby('parcel_number')['MAX_HGT'].transform('max'))
# drop any of the records less than best guess thresholds for main structure
pc_meet_bg_thresh = pc_meet_thresh[(pc_meet_thresh['area_ratio'] >= .95) &
                                   (pc_meet_thresh['height_ratio'] >= .5)].copy()
pc_meet_bg_thresh['n_bld'] = pc_meet_bg_thresh.groupby('parcel_number').transform('size')
pc_meet_bg_thresh_uniq = pc_meet_bg_thresh[pc_meet_bg_thresh['n_bld'] == 1]
pc_bld_l.append(pc_meet_bg_thresh_uniq)

# The concat df has unique parcel record to building footprint pairings
pc_bld = pd.concat(pc_bld_l, axis=0)[['parcel_number', 'val_struct', 'bfid', 'geometry']]

# Proportion of unique matches to possible
unique_match_prop = len(pc_bld)/len(bld_fp_o['parcel_number'].unique())
print('Number of unique tax records matched: {}'.format(len(pc_bld)))
print('Number of tax records: {}'.format(len(bld_fp_o['parcel_number'].unique())))
print('Proportion of unique sf tax records matched: {}'.format(unique_match_prop))

Number of unique tax records matched: 93797
Number of tax records: 94016
Proportion of unique sf tax records matched: 0.9976706092579987


Now we want to aggregate tax records that represent buildings with many units and disaggregate tax records that represent parcels with many buildings. We filtered the sf residential type that need to be disaggregated in a way that likely dropped garages, so we will work separately on those. 

In [395]:
# Remaining cases
pc_complex = bld_fp_o[~bld_fp_o['parcel_number'].isin(pc_bld['parcel_number'])]
pc_complex['stories_n'] = pc_complex['bld_code_rest'].apply(lambda x: x[0])
pc_complex = pc_complex.set_index(['bld_type', 'stories_n']).join(main_thresh)

# List of lower confidence single main building structures
# to concat after processing
pc_complex_l = []

# Some have multiple footprints because they didn't make the cut
# for the "best guess" main thresholds. We can relax the area 
# threshold for these by filtering for properties in the height
# range and then using the combined area & height ratio to feel
# more confident that we're dropping garages and other
# appertunant structures
pc_complex_main = pc_complex[(pc_complex['MAX_HGT'] >= pc_complex['height_min']) &
                             (pc_complex['MAX_HGT'] <= pc_complex['height_upper'])].copy()
pc_complex_main['n_bld'] = pc_complex_main.groupby('parcel_number').transform('size')
pc_meet_thresh_unique = pc_complex_main[pc_complex_main['n_bld'] == 1].copy()

pc_complex_l.append(pc_meet_thresh_unique)

# Do ratio checks (ignoring best guess thresholds) for pc_complex and
# let's see if we can identify the main structures or have to treat all
# as if they are apartment complexes
pc_no_thresh = pc_complex[~pc_complex['parcel_number'].isin(pc_meet_thresh_unique['parcel_number'])].copy()

# compare the ratio of each building's m2_bld to the group
# of buildings in that parcel
pc_no_thresh['area_ratio'] = (pc_no_thresh['m2_bld'] / 
                                pc_no_thresh.groupby('parcel_number')['m2_bld'].transform('max'))
pc_no_thresh['height_ratio'] = (pc_no_thresh['MAX_HGT'] / 
                                  pc_no_thresh.groupby('parcel_number')['MAX_HGT'].transform('max'))
# drop any of the records less than best guess thresholds for main structure
pc_no_thresh_bg = pc_no_thresh[(pc_no_thresh['area_ratio'] >= .95) &
                                   (pc_no_thresh['height_ratio'] >= .5)].copy()
pc_no_thresh_bg['n_bld'] = pc_no_thresh_bg.groupby('parcel_number').transform('size')
pc_no_thresh_bg_uniq = pc_no_thresh_bg[pc_no_thresh_bg['n_bld'] == 1].copy()

pc_complex_l.append(pc_no_thresh_bg_uniq)

# Finally, if all the bld_fp linked to the parcel have area < 10, we'll just
# keep the building footprint with the largest area
small_area_pc_check = pc_no_thresh[~pc_no_thresh['parcel_number'].isin(pc_no_thresh_bg_uniq['parcel_number'])]
small_area_pc = small_area_pc_check.groupby('parcel_number')['m2_bld'].transform(lambda x: (x < 10).all())
max_area = small_area_pc_check.groupby('parcel_number')['m2_bld'].transform('max')
is_largest = (small_area_pc_check['m2_bld'] == max_area) & small_area_pc
pc_no_thresh_lgst = small_area_pc_check[is_largest]

pc_complex_l.append(pc_no_thresh_lgst)

pc_bld_low_conf = pd.concat(pc_complex_l, axis=0)[['parcel_number', 'val_struct', 'bfid', 'geometry']]

# For the rest, we want to assume the parcel record has to be
# split up across all linked buildings. For those that
# have an address connection (or better yet, the same tax record geometry)
# as those in the tax records, we want to aggregate the record before
# doing the splitting
pc_disagg = pc_complex[~pc_complex['parcel_number'].isin(pc_bld_low_conf['parcel_number'])].reset_index()
pc_disagg = pc_disagg[['parcel_number', 'val_struct', 'bfid', 'geometry']]

/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


We need to check if we are missing any tax records from `assess_linked` in our new datasets of tax records linked to one main building or tax records to disaggregate across structures. These can be missing because their building footprint is missing from `bld_fp`. Let's check out what's happening

In [398]:
# Tax records linked to one main building
pc_bld_main = pd.concat([pc_bld, pc_bld_low_conf], axis=0).reset_index(drop=True)

In [322]:
# These are records we can link to parcels but not to building footprints, 
# at least with the overlay
dir_mat_missed = direct_matches[~(direct_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
                                ~(direct_matches['parcel_number'].isin(pc_disagg['parcel_number']))]

# For example, this code will return an empty dataframe
# bld_fp_o[bld_fp_o['parcel_number'].isin(dir_mat_missed['parcel_number'])]

# But also can't find any of these records in the bld_fp data...
# dir_mat_missed[dir_mat_missed['PARCEL_ID'].isin(bld_fp['PARCEL_ID_NUM'])]
# dir_mat_missed[dir_mat_missed['ADDRESS'].isin(bld_fp['ADDRESS'])]

# Simply put, these are missing building footprints. See the following code for
# quick visualization of these instances
# Replace the BRT_ID with samples of BRT_ID from dir_mat_missed
# Some of these have structures but they're missing whereas others
# are vacant. I say we treat the Philly footprints as our baseline
# and treat these as examples of no building...
# We can use these parcel boundaries as a filter to remove NSI
# points inside of them as a sensitivity check

# from shapely.geometry import box
# import matplotlib.pyplot as plt
# temp = parcel[parcel['BRT_ID'] == '291124701']
# bbox = temp.total_bounds
# window = box(*bbox).buffer(.0001)

# fig, ax = plt.subplots()

# temp2 = bld_fp[bld_fp.geometry.intersects(window)]

# if not temp2.empty:
#     temp2.plot(ax=ax)
# temp.plot(ax=ax, color='none', edgecolor='red')

We check the parcels to disaggregate with the records we didn't link up to parcels. Some of these (maybe all) are the units in condos or apartment buildings and need to be aggregated with our parcels to disaggregate. If some of them don't link up, we have to check if we can link the tax record with one of our records in pc_bld_main, which suggests aggregating structure characteristics. Alternatively, we can see if we can link the tax record to a building footprint through a spatial join (through a parcel boundary and/or building footprint). Once we have no more stones unturned, we will have our set of records to disaggregate across structures. We also have to do aggregation in pc_bld_main for condos. 

In [439]:
# Only need value for aggregation and parcel_number for groupby
agg_cols = ['val_struct', 'BRT_ID']

# Link assess_no_link to parcel
assess_sp_link = gpd.sjoin(assess_no_link,
                           parcel, 
                           predicate='within')

assess_sp_link = assess_sp_link.loc[:, agg_cols].copy()

# Find parcel matches in pc_bld_main for aggregation
pc_agg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_bld_main['parcel_number'])]
# Same for pc_disagg
pc_disagg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_disagg['parcel_number'])]

# Get corresponding records from each of pc_agg_match & pc_disagg_match 
# so we can do aggregation (and subsequent disagg where needed)
pc_agg_add = pc_bld_main[pc_bld_main['parcel_number'].isin(pc_agg_match['BRT_ID'])].copy()
pc_agg_add = pc_agg_add.rename(columns={'parcel_number': 'BRT_ID'})
# Add relevant agg characteristics to dataframe
pc_agg_add = pc_agg_add.loc[:, agg_cols].copy()
# Then concat them
pc_agg_proc = pd.concat([pc_agg_match, pc_agg_add], axis=0)

# Repeat for pc_disagg_match
pc_disagg_add = pc_disagg[pc_disagg['parcel_number'].isin(pc_disagg_match['BRT_ID'])].copy()
pc_disagg_add = pc_disagg_add.rename(columns={'parcel_number': 'BRT_ID'})
pc_disagg_add = pc_disagg_add.loc[:, agg_cols].copy()
pc_disagg_proc = pd.concat([pc_disagg_match, pc_disagg_add], axis=0)

# Aggregate structure values, make dict, replace vals in main df
pc_agg_sum = pc_agg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_agg_dict = dict(zip(pc_agg_sum['BRT_ID'], pc_agg_sum['val_struct']))
a_mask = pc_bld_main['parcel_number'].isin(pc_agg_sum['BRT_ID'])
pc_bld_main.loc[a_mask, 'val_struct'] = pc_bld_main.loc[a_mask, 'parcel_number'].map(pc_agg_dict)

pc_disagg_sum = pc_disagg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_disagg_dict = dict(zip(pc_disagg_sum['BRT_ID'], pc_disagg_sum['val_struct']))
d_mask = pc_disagg['parcel_number'].isin(pc_disagg_sum['BRT_ID'])
pc_disagg.loc[d_mask, 'val_struct'] = pc_disagg.loc[d_mask, 'parcel_number'].map(pc_disagg_dict)

Drop structures with no value. Several checks suggest these are demolished structures/vacant lots. Some checks don't suggest this, but it's such a small number we can remove in our "best guess" inventory. 

Next, merge other structure characteristics into pc_bld_main and pc_disagg (including our method for assigning RES1 and RES3 best guesses). Disaggregate for parcels with many structures and merge into pc_bld_main (need a new dataframe for these records). 

Finally, we want to check what tax records were lost along the way in this processing. Can we directly do tax record in building footprint spatial joins to recover? After leaving no stone unturned, we will call our residential inventory final and write it out with a parsimonious set of columns. 

In [451]:
inv_cols = ['parcel_number', 'basements', 'number_stories',
            'bld_type', 'bld_code_rest', 'val_struct', 
            'bfid', 'geometry']
assess_merge_cols = ['parcel_number', 'basements', 'number_stories',
                     'bld_type', 'bld_code_rest']

# Merge in characteristics and drop no val records
pc_bld_main_inv = pc_bld_main.merge(assess_res[assess_merge_cols],
                                    on='parcel_number')
pc_bld_main_inv = pc_bld_main_inv.loc[pc_bld_main_inv['val_struct'] > 0].copy()

pc_disagg_inv = pc_disagg.merge(assess_res[assess_merge_cols],
                                on='parcel_number')
pc_disagg_inv = pc_disagg_inv.loc[pc_disagg_inv['val_struct'] > 0].copy()

# Add RES1 & RES3 based on bld_type mappings and adjacent building processing
# All in pc_disagg are RES3


# Divide structure value in pc_disagg_inv by proportion of sq ft
# a building takes up for those matched to the same parcel. 
# Could do evenly by number of structures but $/sq ft is closer
# to the NSI method and seems like a good benchmark

In [ ]:
pc_agg_proc.groupby('BRT_ID')['val_struct'].sum

BRT_ID
084050706      427400.0
084050708      420800.0
084050710      420800.0
084050712      420800.0
084050714      420800.0
                ...    
888480000    18204300.0
888520216    42641200.0
888520668    39981300.0
888600030     1851200.0
888800326     2034200.0
Name: val_struct, Length: 466, dtype: float64

In [410]:
pc_disagg_proc.groupby('BRT_ID')['val_struct'].sum()

BRT_ID
881000222    10322100.0
881113406    29169000.0
881822814    98158860.0
888000118     2209500.0
888084113    31150600.0
888101741    10608800.0
888101770    11496600.0
888151623    15049000.0
888210145    46161900.0
888260020     4139600.0
888260080    29076200.0
888290002    43836000.0
888290694      725500.0
888380114    46686900.0
Name: val_struct, dtype: float64

In [383]:
pc_bld_main.columns

Index(['parcel_number', 'bfid', 'geometry', 'bld_type', 'location',
       'building_code_description_new'],
      dtype='object')

In [384]:
pc_disagg.columns

Index(['bld_type', 'stories_n', 'BRT_ID', 'PARCEL_ID', 'ADDRESS_1',
       'parcel_number', 'bld_code_rest', 'building_code_description_new',
       'basements', 'unit', 'BIN', 'PARCEL_ID_NUM', 'ADDRESS_2', 'MAX_HGT',
       'geometry', 'm2_bld', 'bfid', 'n_parcels', 'n_bld', 'height_min',
       'height_upper', 'area_min'],
      dtype='object')

In [365]:
# Let's look through the addresses in both pc_bld_main and pc_disagg for matches in 
# assess_no_link to see where we have to aggregate
# After aggregation, we will only disaggregate those in pc_disagg to
# the different building footprints linked up
pc_bld_main[pc_bld_main['location'].isin(assess_no_link['location'])].head()

,parcel_number,bfid,geometry,bld_type,location,building_code_description_new
210,888303930,96975,"POLYGON ((-75.17858 39.94167, -75.17857 39.941...",RES CONDO,2125 CHRISTIAN ST,None
453,888303900,96995,"POLYGON ((-75.17847 39.94165, -75.17846 39.941...",RES CONDO,2121 CHRISTIAN ST,None
1196,888087003,101214,"POLYGON ((-75.17301 39.95163, -75.17295 39.951...",RES CONDO,1920-22 CHESTNUT ST,None
1322,888154066,78306,"POLYGON ((-75.18267 39.97283, -75.18263 39.973...",RES CONDO,2810 POPLAR ST,ROW TYPICAL
1643,888304126,86016,"POLYGON ((-75.1761 39.93962, -75.17609 39.9396...",RES CONDO,1021 S 20TH ST,None


In [364]:
pc_disagg[pc_disagg['ADDRESS_1'].isin(assess_no_link['location'])].head()

,bld_type,stories_n,BRT_ID,PARCEL_ID,ADDRESS_1,parcel_number,bld_code_rest,building_code_description_new,basements,unit,...,ADDRESS_2,MAX_HGT,geometry,m2_bld,bfid,n_parcels,n_bld,height_min,height_upper,area_min
110,ROW,2,888000118,90983,4019 HAVERFORD AVE,888000118,"[2, STY, MASONRY]",None,A,1,...,4019 HAVERFORD AVE,NaN,"POLYGON ((-75.2036 39.96326, -75.20357 39.9632...",123.691884,53391,1,2,14.0,38.575821,10.0
111,ROW,2,888000118,90983,4019 HAVERFORD AVE,888000118,"[2, STY, MASONRY]",None,A,1,...,4019 HAVERFORD AVE,NaN,"POLYGON ((-75.20356 39.963, -75.20351 39.96301...",61.891712,53392,1,2,14.0,38.575821,10.0
112,RES CONDO,4,888101741,94147,2034 ARCH ST,888101741,"[4, STY, MASONRY]",None,None,KLW,...,2034 ARCH ST,57.0,"POLYGON ((-75.17431 39.95544, -75.1743 39.9554...",82.046035,102007,1,10,35.0,82.014922,10.0
113,RES CONDO,4,888101741,94147,2034 ARCH ST,888101741,"[4, STY, MASONRY]",None,None,KLW,...,2034 ARCH ST,59.0,"POLYGON ((-75.17398 39.95535, -75.17397 39.955...",77.254269,102006,1,10,35.0,82.014922,10.0
114,RES CONDO,4,888101741,94147,2034 ARCH ST,888101741,"[4, STY, MASONRY]",None,None,KLW,...,2034 ARCH ST,57.0,"POLYGON ((-75.17429 39.95554, -75.17428 39.955...",82.099097,102014,1,10,35.0,82.014922,10.0


In [360]:
assess_no_link[assess_no_link['location'].isin(pc_disagg['ADDRESS_1'])][['parcel_number', 'bld_type']]

,parcel_number,bld_type
893,888210247,RES CONDO
1054,888084085,RES CONDO
3574,888210315,RES CONDO
3603,888210171,RES CONDO
3711,888210375,RES CONDO
...,...,...
112936,888210142,RES CONDO
113426,888210143,RES CONDO
113532,888210188,RES CONDO
114397,888210109,RES CONDO


In [366]:
assess_no_link[assess_no_link['location'].isin(pc_bld_main['location'])][['parcel_number', 'bld_type']].head()

,parcel_number,bld_type
320,888036170,RES CONDO
329,888081298,RES CONDO
347,888152364,RES CONDO
356,888073200,RES CONDO
360,888520584,RES CONDO


In [368]:
assess_no_link[(~assess_no_link['location'].isin(pc_bld_main['location'])) &
                (~assess_no_link['location'].isin(pc_disagg['ADDRESS_1']))][['parcel_number', 'bld_type']]

,parcel_number,bld_type
330,888092240,RES CONDO
351,881001097,SEMI/DET
527,888380010,RES CONDO
536,888111818,RES CONDO
542,888083398,RES CONDO
...,...,...
122160,888000611,S/D W/D GAR
122162,888000598,S/D W/D GAR
122163,888000602,S/D W/D GAR
122347,888305116,ROW CONV/APT


Now that we have our tax records linked to building foorprints, we can finalize our initialization of the inventory by getting the characteristics set up. This includes occupancy type code, structure value, stories, and basement type. We should have "best guess" versions of the columns as well as different alternatives based on defensible assumptions to test as sensitivity analyses. For example, we think some bld_type correspond to RES1 if they don't touch another structure but we can do a sensitivity check treating them as RES3 (if you think a row home should be RES3 no matter what). 

Now, we look at the subset of properties that have mixed RES1/RES3 *and* are the same size as row homes. We create a new column that indicates which of those touch at least one other building footprint. 

In [ ]:
touches_df = gpd.sjoin(gdf, gdf, how='inner', predicate='touches')
gdf['touches'] = gdf.index.isin(touches_df.index).astype(int)

## Process vulnerability

## Process reference data

## Hazard

In [ ]:
# HAZ_DIR_UZ
# HAZ_FILEN (need to modify with a wildcard for ensemble numbers from 01 to 50 - haz_nens)
# We need a function that turns any of the files into a depth grid
# We don't necessarily have to save these - might not want all that data
# I do want to do this for the "best estimate" though
# I only want the array part of the others
HAZ_CRS

In [ ]:
HAZ_DIR_UZ